<a href="https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content_hash_id's 30-day snapshot (imp_last30/prev30, clk_last30, pos_last30, ctr_last30 window), used to flag pages whose position is worse than the panel median (pos_last30 > 81.7).

Time window for development: month = 2026-03 (mid-panel) Sealed test month: 2026-06 (_sample table) — mechanics only, not label logic

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| ctr_last30 | Feature | known at decision time, describes engagement not position |
| visible_queries | Feature | known prior — query diversity signal |
| top_query_share | Feature | known prior — concentration signal |
| rare_share | Feature | known prior |
| anon_share | Feature | known prior |
| pos_last30 | Label source | used to build label — excluded as a feature to avoid leakage |
| content_hash_id, client_hash_id | Context | identifiers, not model input |
| imp_last30, clk_last30 | Excluded | same-window as label; risk of leaking position-linked signal |

In [ ]:
import duckdb
from google.colab import userdata
from huggingface_hub import hf_hub_download

# 1. Retrieve HF Token from Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Download March 2026 file using exact path from output
file_path = "fact_content_daily_performance/month=2026-03/data_0.parquet"

local_parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=file_path,
    repo_type="dataset",
    token=hf_token
)

# 3. Query with DuckDB
con = duckdb.connect()
df = con.execute(f"SELECT * FROM '{local_parquet_path}' LIMIT 5").df()

print("✅ Connection successful!")
display(df)

✅ Connection successful!


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [ ]:
feature_df = con.execute(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS avg_gsc_position,                     -- Feature 1
        COALESCE(SUM(gsc_impressions), 0) AS total_gsc_impressions,     -- Feature 2
        COALESCE(SUM(gsc_clicks), 0) AS total_gsc_clicks,               -- Feature 3
        COALESCE(SUM(ga4_pageviews), 0) AS total_ga4_pageviews,         -- Feature 4
        COALESCE(SUM(ga4_sessions), 0) AS total_ga4_sessions            -- Feature 5
    FROM '{local_parquet_path}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

display(feature_df.head())

,content_hash_id,avg_gsc_position,total_gsc_impressions,total_gsc_clicks,total_ga4_pageviews,total_ga4_sessions
0,content_7a105f548d9c6916,7.209549,6523.0,7.0,1.0,1.0
1,content_a3ea9792f793ec72,2.987198,453.0,0.0,0.0,0.0
2,content_36c36abc7650d7af,6.724039,5630.0,6.0,6.0,3.0
3,content_a7da352b73b02668,7.244844,4944.0,13.0,2.0,2.0
4,content_1855a661b4d36130,4.209227,429.0,1.0,2.0,2.0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT CONCAT(content_hash_id, '||', CAST(report_date AS STRING))) AS unique_grain_rows
    FROM '{local_parquet_path}'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_rows
0,9841378,9841378


In [ ]:
con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM '{local_parquet_path}'
""").df()

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT_IF(gsc_data_available IS TRUE) AS available_rows,
        ROUND(COUNT_IF(gsc_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS availability_pct
    FROM '{local_parquet_path}'
    WHERE gsc_data_available IS TRUE
""").df()

,total_rows,available_rows,availability_pct
0,3611061,3611061.0,100.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Directional correlation only, not causation — low query diversity doesn't cause ranking drop, model just uses it as a signal
Missing imp_prev30 data limits deeper historical pattern analysis beyond 60 days
Class 0 (non-decay) recall is weaker (67%) vs Class 1 recall (97%)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.